# ForestClustering: synthetic dataset — scale, quality, robustness

**Notebook goals:**
1. Generate a dataset with **known structure** (5 clusters) and mixed feature types
2. Show that ForestClusterer correctly assigns weights to correlated features (1/G)
3. Compare with sklearn algorithms by ARI, Silhouette, and time
4. Quantitatively verify robustness to outliers
5. Demonstrate scalability up to 100K observations

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import (KMeans, MiniBatchKMeans, DBSCAN,
                              AgglomerativeClustering)
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.compose import ColumnTransformer
from sklearn.neighbors import NearestNeighbors

import time
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), ''))

from forest_clustering import ForestClusterer

sns.set_theme(style='whitegrid', palette='husl', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120})

PALETTE5 = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']
ALGO_COLOR = {
    'ForestClusterer': '#2196F3',
    'KMeans':          '#FF9800',
    'MiniBatchKMeans': '#FFC107',
    'AgglomerativeClustering': '#4CAF50',
    'DBSCAN':          '#9C27B0',
}

## 1. Data generator

We create a dataset with:
- 3 **continuous** features — informative for clusters
- 2 **binary** features — probability depends on cluster
- 2 **categorical** features (4 and 5 categories) — distribution depends on cluster
- 2 **correlated** features (copies of cont_1, cont_2 + small noise) — to verify 1/G weighting
- 3 **noise** features — carry no information

In [ ]:
def make_mixed_dataset(n_samples: int, n_clusters: int = 5,
                       outlier_fraction: float = 0.0, seed: int = 42):
    """Generate mixed-type clustering benchmark dataset.

    Returns (df, true_labels).  true_labels == -1 marks outliers.
    Feature layout:
        cont_1/2/3      — informative continuous
        binary_1/2      — informative binary
        cat_1 (A-D)     — informative categorical, 4 categories
        cat_2 (v-z)     — informative categorical, 5 categories
        corr_1/2        — strongly correlated with cont_1/2 (triggers 1/G weighting)
        noise_1/2/3     — pure noise
    """
    rng = np.random.default_rng(seed)
    n_out = int(n_samples * outlier_fraction)
    n_core = n_samples - n_out

    # Cluster sizes
    sizes = np.full(n_clusters, n_core // n_clusters, dtype=int)
    sizes[:n_core % n_clusters] += 1

    # Well-separated cluster centres in 3D
    centers = rng.uniform(-7, 7, (n_clusters, 3))
    # ensure minimum separation of 4 between centres
    for _ in range(200):
        ok = True
        for i in range(n_clusters):
            for j in range(i+1, n_clusters):
                if np.linalg.norm(centers[i]-centers[j]) < 4:
                    centers[j] = rng.uniform(-7, 7, 3)
                    ok = False
        if ok:
            break

    cont_parts, bin_parts, cat_parts, labels = [], [], [], []

    for k, n_k in enumerate(sizes):
        labels.extend([k] * n_k)
        cont_parts.append(rng.normal(centers[k], 0.9, (n_k, 3)))

        # binary: monotone p across clusters
        p1 = 0.1 + 0.8 * k / (n_clusters - 1)
        p2 = 0.9 - 0.8 * k / (n_clusters - 1)
        bin_parts.append(rng.binomial(1, [p1, p2], (n_k, 2)).astype(float))

        # cat_1: dominant category = k % 4, background 0.1/3
        d1 = np.full(4, 0.1/3)
        d1[k % 4] = 0.7
        d1 /= d1.sum()
        # cat_2: dominant = k % 5
        d2 = np.full(5, 0.1/4)
        d2[k % 5] = 0.6
        d2 /= d2.sum()
        cat_parts.append(np.column_stack([rng.choice(4, n_k, p=d1),
                                           rng.choice(5, n_k, p=d2)]))

    X_cont = np.vstack(cont_parts)
    X_bin  = np.vstack(bin_parts)
    X_cat  = np.vstack(cat_parts)
    X_corr = X_cont[:, :2] + rng.normal(0, 0.08, (n_core, 2))  # highly correlated
    X_noise = rng.normal(0, 4, (n_core, 3))

    y_core = np.array(labels)

    # Outliers: uniform over extended range
    if n_out > 0:
        X_out_cont  = rng.uniform(-22, 22, (n_out, 3))
        X_out_bin   = rng.binomial(1, 0.5, (n_out, 2)).astype(float)
        X_out_cat   = np.column_stack([rng.integers(0, 4, n_out),
                                        rng.integers(0, 5, n_out)])
        X_out_corr  = rng.uniform(-22, 22, (n_out, 2))
        X_out_noise = rng.normal(0, 4, (n_out, 3))

        X_cont  = np.vstack([X_cont,  X_out_cont])
        X_bin   = np.vstack([X_bin,   X_out_bin])
        X_cat   = np.vstack([X_cat,   X_out_cat])
        X_corr  = np.vstack([X_corr,  X_out_corr])
        X_noise = np.vstack([X_noise, X_out_noise])
        y_all   = np.concatenate([y_core, np.full(n_out, -1)])
    else:
        y_all = y_core

    X_all = np.hstack([X_cont, X_bin, X_cat.astype(float), X_corr, X_noise])
    perm  = rng.permutation(n_samples)
    X_all, y_all = X_all[perm], y_all[perm]

    COL_NAMES = ['cont_1','cont_2','cont_3',
                 'binary_1','binary_2',
                 'cat_1','cat_2',
                 'corr_1','corr_2',
                 'noise_1','noise_2','noise_3']

    df = pd.DataFrame(X_all, columns=COL_NAMES)
    df['cat_1'] = df['cat_1'].astype(int).map({0:'A',1:'B',2:'C',3:'D'})
    df['cat_2'] = df['cat_2'].astype(int).map({0:'v',1:'w',2:'x',3:'y',4:'z'})
    df['binary_1'] = df['binary_1'].astype(int)
    df['binary_2'] = df['binary_2'].astype(int)

    return df, y_all

# Generate a "clean" dataset (n=10K for comparison) and a "polluted" one (5% outliers)
N_COMPARE = 10_000
N_CLUSTERS = 5

df_clean, y_clean = make_mixed_dataset(N_COMPARE, n_clusters=N_CLUSTERS, seed=42)
df_dirty, y_dirty = make_mixed_dataset(N_COMPARE, n_clusters=N_CLUSTERS, outlier_fraction=0.05, seed=42)

print(f"Clean dataset:    {df_clean.shape},  clusters: {sorted(set(y_clean))}")
print(f"Polluted dataset: {df_dirty.shape},  outliers: {(y_dirty==-1).sum()}")
df_clean.head()

In [ ]:
# EDA: feature types and distributions
COL_NAMES = df_clean.columns.tolist()
FEATURE_GROUPS = {
    'Continuous':   ['cont_1', 'cont_2', 'cont_3'],
    'Binary':       ['binary_1', 'binary_2'],
    'Categorical':  ['cat_1', 'cat_2'],
    'Correlated':   ['corr_1', 'corr_2'],
    'Noise':        ['noise_1', 'noise_2', 'noise_3'],
}

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.ravel()

ax_idx = 0
for group, cols in FEATURE_GROUPS.items():
    for col in cols:
        ax = axes[ax_idx]
        if df_clean[col].dtype == object:
            df_clean[col].value_counts().sort_index().plot.bar(ax=ax, color='#4C72B0',
                                                                edgecolor='white')
            ax.tick_params(axis='x', rotation=0)
        else:
            for k in range(N_CLUSTERS):
                mask = y_clean == k
                ax.hist(df_clean.loc[mask, col], bins=30, alpha=0.4,
                        color=PALETTE5[k], label=f'cl {k+1}')
        ax.set_title(f'{col}  [{group}]', fontsize=9, fontweight='bold')
        ax.set_ylabel('')
        ax_idx += 1

for i in range(ax_idx, len(axes)):
    axes[i].set_visible(False)

plt.suptitle('Feature distributions by cluster', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# PCA: true clusters
from sklearn.preprocessing import OrdinalEncoder as OE

X_vis = df_clean.copy()
X_vis['cat_1'] = OE().fit_transform(X_vis[['cat_1']])
X_vis['cat_2'] = OE().fit_transform(X_vis[['cat_2']])
X_scaled = StandardScaler().fit_transform(X_vis.astype(float))
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(8, 6))
for k in range(N_CLUSTERS):
    m = y_clean == k
    ax.scatter(coords[m, 0], coords[m, 1], s=6, alpha=0.5, color=PALETTE5[k], label=f'Cluster {k+1}')
ax.set_title('True clusters (PCA)', fontsize=13, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.legend(markerscale=3, loc='upper right')
plt.tight_layout()
plt.show()

## 2. ForestClusterer: embeddings and feature weights

In [ ]:
t0 = time.perf_counter()
fc = ForestClusterer(
    n_iterations=250,
    n_bins=3,
    quantile_cuts=True,       # quantile cut-points → outlier robustness
    clusterer=AgglomerativeClustering(n_clusters=N_CLUSTERS, metric='precomputed', linkage='average'),
    corr_threshold=0.9,       # conservative threshold: catches only |Spearman|>0.9
    corr_sample_size=5_000,
    random_state=42,
)
fc_labels = fc.fit_predict(df_clean)
fc_time = time.perf_counter() - t0

ari = adjusted_rand_score(y_clean, fc_labels)
sil = silhouette_score(fc.get_embedding().astype(float), fc_labels,
                       metric='hamming', sample_size=2000, random_state=0)
print(f"ForestClusterer  |  ARI={ari:.3f}  Silhouette={sil:.3f}  Time={fc_time:.1f}s")

In [ ]:
# Feature weights
weights = fc.feature_weights_
features = COL_NAMES

GROUP_COLOR = {
    'cont_1': '#E53935', 'cont_2': '#E53935', 'cont_3': '#1565C0',
    'binary_1': '#2E7D32', 'binary_2': '#2E7D32',
    'cat_1': '#6A1B9A', 'cat_2': '#6A1B9A',
    'corr_1': '#E53935', 'corr_2': '#E53935',   # same group as cont_1/2
    'noise_1': '#78909C', 'noise_2': '#78909C', 'noise_3': '#78909C',
}

fig, ax = plt.subplots(figsize=(12, 4))
colors = [GROUP_COLOR.get(f, '#78909C') for f in features]
bars = ax.bar(features, weights, color=colors, edgecolor='white', linewidth=1.5)

ax.axhline(1.0, color='black', ls='--', lw=1, label='weight=1 (independent)')
ax.axhline(0.5, color='red',   ls=':', lw=1.2, label='weight=0.5 (group of 2)')

legend_patches = [
    Patch(color='#E53935', label='Correlated (cont_1, cont_2, corr_1, corr_2)'),
    Patch(color='#1565C0', label='Continuous (cont_3)'),
    Patch(color='#2E7D32', label='Binary'),
    Patch(color='#6A1B9A', label='Categorical'),
    Patch(color='#78909C', label='Noise'),
]
ax.legend(handles=legend_patches, loc='upper right', fontsize=9)

for bar, w in zip(bars, weights):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{w:.2f}', ha='center', va='bottom', fontsize=9)

ax.set_ylim(0, 1.3)
ax.set_title('Feature weights (1/G for correlated groups)', fontweight='bold', fontsize=13)
ax.set_ylabel('Weight in feature sampling')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

print("Correlated features detected:", [f for f,w in zip(features,weights) if w < 0.99])

In [ ]:
# Distance matrix visualisation (subsample 300 obs, sorted by cluster)
SAMPLE_VIZ = 300
sample_idx = []
for k in range(N_CLUSTERS):
    idxs = np.where(fc_labels == k)[0]
    sample_idx.extend(np.random.choice(idxs, min(SAMPLE_VIZ // N_CLUSTERS, len(idxs)), replace=False))
sample_idx = np.array(sample_idx)

E_sub = fc.get_embedding()[sample_idx]
from forest_clustering import pairwise_hamming
D_sub = pairwise_hamming(E_sub)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(D_sub, cmap='YlOrRd', vmin=0, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, label='Hamming distance')

# Cluster boundaries
n_per_cluster = len(sample_idx) // N_CLUSTERS
for i in range(1, N_CLUSTERS):
    ax.axhline(i * n_per_cluster - 0.5, color='white', lw=1.5)
    ax.axvline(i * n_per_cluster - 0.5, color='white', lw=1.5)

ax.set_title('Pairwise distance matrix (300 obs, sorted by cluster)',
             fontweight='bold')
ax.set_xlabel('Observation')
ax.set_ylabel('Observation')
plt.tight_layout()
plt.show()

## 3. Comparison with sklearn algorithms

**Conditions:**
- n=10K, 5 clusters, no outliers
- For sklearn algorithms: OneHotEncoder (cat) + StandardScaler (num), ~18 features total
- ForestClusterer: raw data, n_iterations=250

In [ ]:
# Preprocessing for sklearn
num_cols = ['cont_1','cont_2','cont_3','corr_1','corr_2','noise_1','noise_2','noise_3']
cat_cols = ['cat_1','cat_2']
bin_cols = ['binary_1','binary_2']  # already numeric

prep = ColumnTransformer([
    ('num', StandardScaler(), num_cols + bin_cols),
    ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), cat_cols),
])
X_prep = prep.fit_transform(df_clean)
print(f"After OHE+Scale: {X_prep.shape[1]} features")

# eps for DBSCAN
nn = NearestNeighbors(n_neighbors=10).fit(X_prep)
kd, _ = nn.kneighbors(X_prep)
eps_auto = np.percentile(kd[:, -1], 90)
print(f"DBSCAN eps: {eps_auto:.3f}")

In [ ]:
results_clean = {}

# ForestClusterer (already computed, quantile_cuts=True, corr_threshold=0.9)
results_clean['ForestClusterer'] = {
    'labels': fc_labels, 'time': fc_time, 'mixed_types': True
}

# KMeans
t0 = time.perf_counter()
km_l = KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=42).fit_predict(X_prep)
results_clean['KMeans'] = {'labels': km_l, 'time': time.perf_counter()-t0, 'mixed_types': False}

# MiniBatchKMeans
t0 = time.perf_counter()
mbkm_l = MiniBatchKMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=5).fit_predict(X_prep)
results_clean['MiniBatchKMeans'] = {'labels': mbkm_l, 'time': time.perf_counter()-t0, 'mixed_types': False}

# Agglomerative (Ward, Euclidean)
t0 = time.perf_counter()
agg_l = AgglomerativeClustering(n_clusters=N_CLUSTERS, linkage='ward').fit_predict(X_prep)
results_clean['AgglomerativeClustering'] = {'labels': agg_l, 'time': time.perf_counter()-t0, 'mixed_types': False}

# DBSCAN (on subsample due to time)
SUB = 3_000
sub_idx = np.random.RandomState(0).choice(N_COMPARE, SUB, replace=False)
t0 = time.perf_counter()
db_l_sub = DBSCAN(eps=eps_auto, min_samples=15).fit_predict(X_prep[sub_idx])
db_time = time.perf_counter()-t0
# Propagate labels via KNN
nn_full = NearestNeighbors(n_neighbors=1).fit(X_prep[sub_idx])
_, nn_idx = nn_full.kneighbors(X_prep)
db_l = db_l_sub[nn_idx[:, 0]]
results_clean['DBSCAN'] = {'labels': db_l, 'time': db_time, 'mixed_types': False}
print(f"DBSCAN (subsample {SUB}): {len(set(db_l_sub)-{-1})} clusters")

In [ ]:
cmp_rows = []
for name, res in results_clean.items():
    lbl = res['labels']
    mask = lbl >= 0
    n_cl = len(set(lbl) - {-1})
    sil = silhouette_score(X_prep[mask], lbl[mask], sample_size=2000, random_state=0)           if n_cl > 1 and mask.sum() > n_cl else float('nan')
    ari = adjusted_rand_score(y_clean, lbl)
    noise_pct = (~mask).mean()

    cmp_rows.append({
        'Algorithm': name,
        'Clusters': n_cl,
        'ARI': round(ari, 3),
        'Silhouette': f'{sil:.3f}' if not np.isnan(sil) else '—',
        'Noise %': f'{noise_pct:.1%}',
        'Time, s': f'{res["time"]:.2f}',
        'Mixed types': '✓' if res['mixed_types'] else '✗',
    })

cmp_df = pd.DataFrame(cmp_rows).set_index('Algorithm')
cmp_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

algo_names = list(results_clean.keys())
aris  = [adjusted_rand_score(y_clean, results_clean[n]['labels']) for n in algo_names]
times = [results_clean[n]['time'] for n in algo_names]
colors = [ALGO_COLOR.get(n, '#607D8B') for n in algo_names]

# ARI
ax = axes[0]
bars = ax.barh(algo_names, aris, color=colors, edgecolor='white', linewidth=1.5)
ax.set_xlim(0, 1.05)
for bar, val in zip(bars, aris):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.3f}',
            va='center', fontsize=10)
ax.set_title('Adjusted Rand Index', fontweight='bold', fontsize=13)
ax.set_xlabel('ARI (higher is better)')
ax.axvline(1.0, color='gray', ls='--', lw=1)

# Time
ax = axes[1]
bars = ax.barh(algo_names, times, color=colors, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, times):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.2f}s',
            va='center', fontsize=10)
ax.set_title('Training time, sec', fontweight='bold', fontsize=13)
ax.set_xlabel('Seconds')

plt.suptitle(f'Algorithm comparison  (n={N_COMPARE:,}, {N_CLUSTERS} clusters)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Outlier robustness

We add **10% outliers** (random points in [-22, 22], far outside the clusters).

**Why ForestClusterer is robust:**
- `quantile_cuts=True` — cut-points are taken from data quantiles, not from [min, max].
  With 10% outliers, ~90% of cut-points fall in the "core" cluster range.
- Hamming distance is bounded in [0, 1] — outliers don't distort the metric.
- We use **HDBSCAN** (density-based clustering): it automatically marks
  outliers as noise (-1) instead of forcing them into clusters.

KMeans assigns all points to clusters, so its centroids shift toward outliers.

In [ ]:
from sklearn.cluster import HDBSCAN

clean_mask = y_dirty != -1
X_prep_dirty = prep.transform(df_dirty)

# ForestClusterer + HDBSCAN: automatically labels outliers as -1
t0 = time.perf_counter()
fc_d = ForestClusterer(
    n_iterations=250, n_bins=3, quantile_cuts=True,
    clusterer=HDBSCAN(metric='precomputed', min_cluster_size=80),
    corr_threshold=0.9, corr_sample_size=5_000, random_state=42,
)
lbl_fc_d = fc_d.fit_predict(df_dirty)
t_fc_d = time.perf_counter()-t0

# KMeans: must assign ALL points to N_CLUSTERS clusters
t0 = time.perf_counter()
lbl_km_d = KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=42).fit_predict(X_prep_dirty)
t_km_d = time.perf_counter()-t0

# Agglomerative: same as KMeans — fixed n_clusters
t0 = time.perf_counter()
lbl_agg_d = AgglomerativeClustering(n_clusters=N_CLUSTERS, linkage='ward').fit_predict(X_prep_dirty)
t_agg_d = time.perf_counter()-t0

print(f"FC+HDBSCAN:  {(lbl_fc_d==-1).sum()} noise detected  (true outliers: {(y_dirty==-1).sum()})")
print(f"KMeans:      0 noise (all points assigned to {N_CLUSTERS} clusters)")

In [ ]:
rob_rows = []
y_true_core = y_dirty[clean_mask]  # true labels of clean observations

# ForestClusterer + HDBSCAN: evaluate only on labeled & clean observations
fc_labeled = lbl_fc_d >= 0
eval_m_fc = fc_labeled & clean_mask
ari_fc_d = adjusted_rand_score(y_dirty[eval_m_fc], lbl_fc_d[eval_m_fc]) if eval_m_fc.sum() > 10 else 0.0
pct_labeled = fc_labeled.mean()
rob_rows.append({
    'Algorithm': 'ForestClusterer+HDBSCAN',
    'ARI (clean)': round(adjusted_rand_score(y_clean, fc_labels), 3),
    'ARI (dirty, labeled obs)': round(ari_fc_d, 3),
    'Labeled fraction': f'{pct_labeled:.1%}',
    'Noise detected': int((lbl_fc_d == -1).sum()),
})

# KMeans and Agglomerative: evaluate on clean_mask
for name, lbl_d, lbl_c in [
    ('KMeans', lbl_km_d, results_clean['KMeans']['labels']),
    ('AgglomerativeClustering', lbl_agg_d, results_clean['AgglomerativeClustering']['labels']),
]:
    ari_c = adjusted_rand_score(y_clean, lbl_c)
    ari_d = adjusted_rand_score(y_true_core, lbl_d[clean_mask])
    rob_rows.append({
        'Algorithm': name,
        'ARI (clean)': round(ari_c, 3),
        'ARI (dirty, labeled obs)': round(ari_d, 3),
        'Labeled fraction': '100%',
        'Noise detected': 0,
    })

rob_df = pd.DataFrame(rob_rows).set_index('Algorithm')
rob_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

algos_rob = ['ForestClusterer+HDBSCAN', 'KMeans', 'AgglomerativeClustering']
x = np.arange(len(algos_rob))
w = 0.35
aris_c = [rob_df.loc[n, 'ARI (clean)'] for n in algos_rob]
aris_d = [rob_df.loc[n, 'ARI (dirty, labeled obs)'] for n in algos_rob]
colors_bar = ['#2196F3', '#FF9800', '#4CAF50']

ax = axes[0]
b1 = ax.bar(x - w/2, aris_c, w, label='Clean', color=colors_bar, edgecolor='white', alpha=0.9)
b2 = ax.bar(x + w/2, aris_d, w, label='+10% outliers', color=colors_bar, edgecolor='white', alpha=0.5, hatch='//')
for bar, val in [(b, v) for bars, vals in [(b1, aris_c), (b2, aris_d)] for b, v in zip(bars, vals)]:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{val:.3f}',
            ha='center', va='bottom', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(algos_rob, fontsize=9, rotation=10)
ax.set_ylim(0, 1.15); ax.set_ylabel('ARI'); ax.set_title('ARI: clean vs with outliers', fontweight='bold')
ax.legend(); ax.axhline(1.0, color='gray', ls='--', lw=1)

# Δ ARI
ax = axes[1]
deltas = [d - c for c, d in zip(aris_c, aris_d)]
bars = ax.bar(algos_rob, deltas, color=colors_bar, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, deltas):
    ax.text(bar.get_x()+bar.get_width()/2, val - 0.015, f'{val:+.3f}',
            ha='center', va='top', fontsize=11, fontweight='bold', color='white')
ax.axhline(0, color='black', lw=1)
ax.set_ylabel('Δ ARI (dirty − clean)')
ax.set_title('ARI drop when adding 10% outliers', fontweight='bold')
ax.set_xticklabels(algos_rob, fontsize=9, rotation=10)

plt.suptitle('Outlier robustness (10% outliers)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Key finding: ForestClusterer+HDBSCAN isolates outliers and maintains high ARI,")
print("while KMeans and Agglomerative are forced to absorb outliers into clusters.")

## 5. Scalability

In [ ]:
# Embedding time vs n_samples
# (n_iterations=200 fixed, n_features='sqrt', n_bins=3)
N_RANGE = [1_000, 5_000, 10_000, 25_000, 50_000, 100_000]
N_ITER_SCALE = 200

times_fc, times_km = [], []

for n in N_RANGE:
    df_n, _ = make_mixed_dataset(n, n_clusters=5, seed=0)
    X_n = prep.transform(df_n)

    # ForestClusterer: embedding only (no final clustering)
    fc_scale = ForestClusterer(n_iterations=N_ITER_SCALE, n_bins=3,
                                corr_threshold=None, random_state=0)
    t0 = time.perf_counter()
    fc_scale.fit(df_n)
    times_fc.append(time.perf_counter() - t0)

    # KMeans as baseline
    t0 = time.perf_counter()
    KMeans(n_clusters=5, n_init=5, random_state=0).fit(X_n)
    times_km.append(time.perf_counter() - t0)

    print(f"n={n:>7,}  ForestClusterer={times_fc[-1]:.2f}s  KMeans={times_km[-1]:.2f}s")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(N_RANGE, times_fc, 'o-', color='#2196F3', lw=2.5, ms=8, label='ForestClusterer')
ax.plot(N_RANGE, times_km, 's-', color='#FF9800', lw=2.5, ms=8, label='KMeans')
ax.set_xlabel('n_samples')
ax.set_ylabel('Time, s')
ax.set_title('Training time vs n_samples', fontweight='bold', fontsize=13)
ax.legend(fontsize=11)
ax.set_xlim(0, N_RANGE[-1] * 1.05)

ax = axes[1]
ax.plot(N_RANGE, times_fc, 'o-', color='#2196F3', lw=2.5, ms=8, label='ForestClusterer')
ax.plot(N_RANGE, times_km, 's-', color='#FF9800', lw=2.5, ms=8, label='KMeans')
ax.set_yscale('log'); ax.set_xscale('log')
ax.set_xlabel('n_samples (log)')
ax.set_ylabel('Time, s (log)')
ax.set_title('Same — log/log scale', fontweight='bold', fontsize=13)
ax.legend(fontsize=11)

plt.suptitle(f'Scalability  (n_iterations={N_ITER_SCALE})', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Linearity check
import numpy as np
log_n = np.log(N_RANGE)
log_t = np.log(times_fc)
slope = np.polyfit(log_n, log_t, 1)[0]
print(f"Log-log slope (ForestClusterer): {slope:.2f}  (1.0 = linear growth)")

## Summary

| Criterion | ForestClusterer | KMeans / Agglomerative |
|---|---|---|
| **Preprocessing** | not required — works with raw mixed-type data | OHE + Scale required |
| **ARI on synthetic** | comparable or better with mixed types | high ARI on numeric-only |
| **Outlier robustness** | Δ ARI minimal | KMeans drops noticeably |
| **Correlated features** | automatically weighted (1/G) | not accounted for |
| **Scalability** | ~linear growth with n | KMeans is also fast |
| **Flexibility** | plug-in clusterer | fixed algorithm |